# Rural primary-care workforce and infant mortality

This notebook matches RBI Table 16 (2023 rural PHC doctor staffing) with RBI Table 4 (2023 infant mortality rate). The staffing ratio is **doctors in position divided by source-reported required doctors**. The requirement is an administrative PHC staffing benchmark (approximately one doctor per PHC), not a population-based estimate of how many doctors residents need. A ratio of 2 means about twice the benchmark post count, not twice the doctors the population needs.

The analysis is cross-sectional and observational; it is not a causal estimate.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

DATA = Path('../Dataset')
staff = pd.read_csv(DATA / 'state_rural_phc_doctors_2023.csv')
imr = pd.read_csv(DATA / 'state_infant_mortality_rate_2023.csv')
staff.head()

## 1. Validate and harmonise state labels

The workforce table combines Dadra & Nagar Haveli and Daman & Diu; the IMR table labels the same combined observation as Dadra & Nagar Haveli (includes Daman & Diu). Chandigarh has zero required posts, so a ratio is undefined. The all-India aggregate is excluded.

In [ ]:
imr = imr.loc[imr['state_ut'] != 'All India'].copy()
imr['state_key'] = imr['state_ut'].replace({'Dadra & Nagar Haveli (includes Daman & Diu)': 'Dadra & Nagar Haveli & Daman & Diu'})
staff['state_key'] = staff['state_ut']
df = staff.merge(imr[['state_key','infant_mortality_rate_2023_per_1000_live_births']], on='state_key', how='inner', validate='one_to_one')
df['phc_doctor_coverage_ratio'] = df['in_position_rural_phc_doctors'] / df['required_rural_phc_doctors'].where(df['required_rural_phc_doctors'] > 0)
analysis = df.dropna(subset=['phc_doctor_coverage_ratio','infant_mortality_rate_2023_per_1000_live_births']).copy()
analysis[['state_ut','required_rural_phc_doctors','in_position_rural_phc_doctors','phc_doctor_coverage_ratio','infant_mortality_rate_2023_per_1000_live_births']].head()

## 2. Interpreting the staffing benchmark

A ratio of 1.0 means in-position doctors equal the source-reported requirement; below 1.0 means fewer doctors are in position than required; above 1.0 means staffing exceeds that administrative benchmark. The denominator is a count of required PHC posts (approximately one doctor per PHC), **not** population need or population size. Small denominators can create very high ratios: Delhi has 5 required doctors and 16 in position (3.20); Goa has 21 required and 97 in position (4.62).

In [ ]:
valid = analysis.sort_values('phc_doctor_coverage_ratio').copy()
display(valid[['state_ut','required_rural_phc_doctors','in_position_rural_phc_doctors','phc_doctor_coverage_ratio']].round(3))
print('Matched observations:', len(valid))
print('Median staffing ratio:', round(valid.phc_doctor_coverage_ratio.median(), 3))
print('Below reported requirement:', int((valid.phc_doctor_coverage_ratio < 1).sum()), 'of', len(valid))

## 3. Small-denominator sensitivity

To assess whether very small PHC requirement counts dominate the association, the notebook reports the full sample and a restricted sample excluding observations with fewer than 50 required rural PHC doctors. This is a transparent sensitivity threshold, not a formal statistical rule.

In [ ]:
def association_summary(data, label):
    x = data['phc_doctor_coverage_ratio']
    y = data['infant_mortality_rate_2023_per_1000_live_births']
    pr, pp = pearsonr(x, y)
    sr, sp = spearmanr(x, y)
    print(f'{label}: n={len(data)}')
    print(f'  Pearson r={pr:.3f}, p={pp:.4f}')
    print(f'  Spearman rho={sr:.3f}, p={sp:.4f}')

large_denominator = analysis.loc[analysis['required_rural_phc_doctors'] >= 50].copy()
association_summary(analysis, 'All matched states/UTs')
association_summary(large_denominator, 'Restricted sample: required doctors >= 50')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(analysis['phc_doctor_coverage_ratio'], analysis['infant_mortality_rate_2023_per_1000_live_births'], alpha=0.8)
for _, row in analysis.iterrows():
    ax.annotate(row['state_ut'], (row['phc_doctor_coverage_ratio'], row['infant_mortality_rate_2023_per_1000_live_births']), fontsize=7, xytext=(3,3), textcoords='offset points')
ax.axvline(1, linestyle='--', linewidth=1, color='black', label='Reported requirement met')
ax.set_xlabel('In-position rural PHC doctors / required rural PHC doctors, 2023')
ax.set_ylabel('Infant mortality rate per 1,000 live births, 2023')
ax.set_title('Rural PHC staffing benchmark and infant mortality')
ax.legend()
fig.tight_layout()
plt.show()

## 4. Interpretation and limitations

The staffing ratio is a comparison with an administrative PHC post benchmark, not doctors per capita, population coverage or proof of adequate access. It excludes private providers, urban facilities, nurses, specialists outside PHCs and service quality. IMR is also affected by income, maternal education, nutrition, sanitation, fertility, transport and other determinants. The analysis is cross-sectional; small states/UTs may have volatile rates. Missing source markers are not imputed as zero.

The full-sample and restricted-sample coefficients are calculated above from the merged source data. They should be interpreted as unadjusted associations only. A multivariable model requires harmonised state-level socioeconomic covariates with clear reference years; those controls are not inferred or imputed in this notebook.

In [ ]:
# Reproducibility checks
assert staff.state_key.is_unique
assert imr.state_key.is_unique
assert len(analysis) == 34, f'Expected 34 matched observations; found {len(analysis)}'
assert len(large_denominator) == 27, f'Expected 27 observations after threshold; found {len(large_denominator)}'
assert analysis['phc_doctor_coverage_ratio'].notna().all()
assert analysis['infant_mortality_rate_2023_per_1000_live_births'].notna().all()
print('Notebook 06 checks passed.')